# Add Source-Level Retrieval Metrics

This notebook does **not** query Qdrant again. It reads existing evaluation outputs and adds source-level metrics:

- `source_recall_at_3/10/20`: any retrieved item has the gold evidence URL.
- `image_source_recall_at_3/10/20`: any retrieved **image** item has the gold evidence URL.

The goal is to check whether image retrieval finds the correct article/source even when it does not match the exact gold image path.

In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"database", "refined"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "database" / "retrieval_eval_outputs"
DETAILS_CSV = OUTPUT_DIR / "retrieval_results_long.csv"
CLAIM_METRICS_CSV = OUTPUT_DIR / "claim_metrics_long.csv"
SUMMARY_CSV = OUTPUT_DIR / "metrics_summary.csv"

UPDATED_DETAILS_CSV = OUTPUT_DIR / "retrieval_results_long_with_source_metrics.csv"
UPDATED_CLAIM_METRICS_CSV = OUTPUT_DIR / "claim_metrics_long_with_source_metrics.csv"
UPDATED_SUMMARY_CSV = OUTPUT_DIR / "metrics_summary_with_source_metrics.csv"

TOP_K_VALUES = [3, 10, 20]
CHUNKSIZE = 200_000

print("Output dir:", OUTPUT_DIR)
print("Details exists:", DETAILS_CSV.exists(), DETAILS_CSV)
print("Claim metrics exists:", CLAIM_METRICS_CSV.exists(), CLAIM_METRICS_CSV)
print("Summary exists:", SUMMARY_CSV.exists(), SUMMARY_CSV)


Output dir: D:\FactCheckPipeline\database\retrieval_eval_outputs
Details exists: True D:\FactCheckPipeline\database\retrieval_eval_outputs\retrieval_results_long.csv
Claim metrics exists: True D:\FactCheckPipeline\database\retrieval_eval_outputs\claim_metrics_long.csv
Summary exists: True D:\FactCheckPipeline\database\retrieval_eval_outputs\metrics_summary.csv


C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helpers

In [2]:
def normalize_url(value) -> str:
    text = str(value or "").strip().lower()
    return text.rstrip("/")

def add_source_flags(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk = chunk.copy()
    retrieved_url = chunk["url"].fillna("").map(normalize_url)
    gold_url = chunk["gold_text_url"].fillna("").map(normalize_url)
    chunk["source_hit"] = (retrieved_url != "") & (gold_url != "") & (retrieved_url == gold_url)
    chunk["image_source_hit"] = chunk["source_hit"] & chunk["modality"].fillna("").eq("image")
    return chunk

def summarize_claim_group(group: pd.DataFrame) -> dict:
    out = {}
    group = group.sort_values("rank")
    for k in TOP_K_VALUES:
        top = group[group["rank"] <= k]
        out[f"source_recall_at_{k}"] = int(top["source_hit"].any())
        out[f"image_source_recall_at_{k}"] = int(top["image_source_hit"].any())
    first_source_rank = group.loc[group["source_hit"], "rank"].min() if group["source_hit"].any() else 0
    first_image_source_rank = group.loc[group["image_source_hit"], "rank"].min() if group["image_source_hit"].any() else 0
    out["first_source_hit_rank"] = int(first_source_rank or 0)
    out["first_image_source_hit_rank"] = int(first_image_source_rank or 0)
    out["source_mrr_at_3"] = (1 / first_source_rank) if first_source_rank and first_source_rank <= 3 else 0.0
    out["image_source_mrr_at_3"] = (1 / first_image_source_rank) if first_image_source_rank and first_image_source_rank <= 3 else 0.0
    return out


## Add Row-Level Source Flags

`retrieval_results_long.csv` can be large, so this cell streams it by chunks and writes a new file.

In [3]:
if UPDATED_DETAILS_CSV.exists():
    UPDATED_DETAILS_CSV.unlink()

first = True
usecols = None
reader = pd.read_csv(DETAILS_CSV, chunksize=CHUNKSIZE)
for chunk in tqdm(reader, desc="Adding source flags"):
    chunk = add_source_flags(chunk)
    chunk.to_csv(UPDATED_DETAILS_CSV, mode="w" if first else "a", header=first, index=False, encoding="utf-8-sig")
    first = False

print("Wrote", UPDATED_DETAILS_CSV)


Adding source flags: 0it [00:00, ?it/s]

C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\std.py:1181: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:


Adding source flags: 1it [00:16, 16.53s/it]

C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\std.py:1181: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:


Adding source flags: 2it [00:32, 16.27s/it]

Adding source flags: 3it [00:33,  9.42s/it]

Adding source flags: 3it [00:33, 11.30s/it]

Wrote D:\FactCheckPipeline\database\retrieval_eval_outputs\retrieval_results_long_with_source_metrics.csv


## Compute Per-Claim Source Metrics

In [4]:
key_cols = [
    "experiment_id", "refiner", "collection", "image_variant", "use_reranker",
    "id", "claim", "label", "gold_text_url", "gold_image_path",
]
source_cols = [*key_cols, "rank", "source_hit", "image_source_hit"]

parts = []
for chunk in tqdm(pd.read_csv(UPDATED_DETAILS_CSV, usecols=source_cols, chunksize=CHUNKSIZE), desc="Loading flagged rows"):
    parts.append(chunk)
details = pd.concat(parts, ignore_index=True)
print(details.shape)

claim_source_metrics = []
for keys, group in tqdm(details.groupby(key_cols, dropna=False), desc="Per-claim metrics"):
    row = dict(zip(key_cols, keys))
    row.update(summarize_claim_group(group))
    claim_source_metrics.append(row)

claim_source_metrics = pd.DataFrame(claim_source_metrics)
claim_source_metrics.head()


Loading flagged rows: 0it [00:00, ?it/s]

Loading flagged rows: 1it [00:04,  4.06s/it]

Loading flagged rows: 2it [00:08,  4.03s/it]

Loading flagged rows: 3it [00:08,  2.33s/it]

Loading flagged rows: 3it [00:08,  2.79s/it]

...

Per-claim metrics:  99%|█████████▉| 20430/20688 [00:23<00:00, 949.42it/s]

Per-claim metrics:  99%|█████████▉| 20532/20688 [00:23<00:00, 967.44it/s]

Per-claim metrics: 100%|█████████▉| 20641/20688 [00:23<00:00, 1001.72it/s]

Per-claim metrics: 100%|██████████| 20688/20688 [00:23<00:00, 880.42it/s] 

,experiment_id,refiner,collection,image_variant,use_reranker,id,claim,label,gold_text_url,gold_image_path,source_recall_at_3,image_source_recall_at_3,source_recall_at_10,image_source_recall_at_10,source_recall_at_20,image_source_recall_at_20,first_source_hit_rank,first_image_source_hit_rank,source_mrr_at_3,image_source_mrr_at_3
0,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,supported,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,1,0,1,0,1,0,1,0,1.0,0.0
1,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,supported,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,1,0,1,0,1,0,1,0,1.0,0.0
2,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,3,Thượng úy Nguyễn Đức Phước là Điều tra viên th...,refuted,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,0,0,1,0,1,0,7,0,0.0,0.0
3,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,nei,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,0,0,0,0,0,0,0,0,0.0,0.0
4,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,nei,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,0,0,0,0,0,0,0,0,0.0,0.0


## Merge With Existing Claim Metrics

In [5]:
claim_metrics = pd.read_csv(CLAIM_METRICS_CSV)
new_metric_cols = [c for c in claim_source_metrics.columns if c not in key_cols]
updated_claim_metrics = claim_metrics.merge(
    claim_source_metrics[key_cols + new_metric_cols],
    on=key_cols,
    how="left",
)
updated_claim_metrics.to_csv(UPDATED_CLAIM_METRICS_CSV, index=False, encoding="utf-8-sig")
print(updated_claim_metrics.shape)
print("Wrote", UPDATED_CLAIM_METRICS_CSV)
display(updated_claim_metrics.head())


(20688, 34)
Wrote D:\FactCheckPipeline\database\retrieval_eval_outputs\claim_metrics_long_with_source_metrics.csv


,experiment_id,refiner,collection,image_variant,use_reranker,id,claim,label,gold_text_url,gold_image_path,...,source_recall_at_3,image_source_recall_at_3,source_recall_at_10,image_source_recall_at_10,source_recall_at_20,image_source_recall_at_20,first_source_hit_rank,first_image_source_hit_rank,source_mrr_at_3,image_source_mrr_at_3
0,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,nei,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,...,0,0,0,0,0,0,0,0,0.0,0.0
1,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,6,Đường dây lừa đảo này chỉ nhắm mục tiêu vào ng...,nei,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,...,0,0,1,0,1,0,4,0,0.0,0.0
2,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,nei,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,...,0,0,0,0,0,0,0,0,0.0,0.0
3,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,supported,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,...,1,0,1,0,1,0,1,0,1.0,0.0
4,gemini-2.5-flash__fixed_size__clip__reranker_0,gemini-2.5-flash,fixed_size,clip,False,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,supported,https://www.facebook.com/mps.gov/posts/pfbid02...,media/post_1_cmt_img_0.jpg,...,1,0,1,0,1,0,1,0,1.0,0.0


## Update Summary Metrics

In [6]:
summary = pd.read_csv(SUMMARY_CSV)
experiment_cols = ["experiment_id", "refiner", "collection", "image_variant", "use_reranker"]
summary_source_metrics = updated_claim_metrics.groupby(experiment_cols, dropna=False)[new_metric_cols].mean().reset_index()
updated_summary = summary.merge(summary_source_metrics, on=experiment_cols, how="left")
updated_summary = updated_summary.sort_values(["evidence_recall_at_3", "source_recall_at_3", "mrr_at_3"], ascending=False)
updated_summary.to_csv(UPDATED_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print(updated_summary.shape)
print("Wrote", UPDATED_SUMMARY_CSV)
display(updated_summary)


(16, 29)
Wrote D:\FactCheckPipeline\database\retrieval_eval_outputs\metrics_summary_with_source_metrics.csv


,experiment_id,refiner,collection,image_variant,use_reranker,num_claims,evidence_recall_at_3,text_recall_at_3,image_recall_at_3,full_recall_at_3,...,source_recall_at_3,image_source_recall_at_3,source_recall_at_10,image_source_recall_at_10,source_recall_at_20,image_source_recall_at_20,first_source_hit_rank,first_image_source_hit_rank,source_mrr_at_3,image_source_mrr_at_3
0,gpt4o_mini__semantic__clip__reranker_1,gpt4o_mini,semantic,clip,True,1293,0.921887,0.921887,0.000000,0.000000,...,0.825986,0.000000,0.888631,0.000000,0.904872,0.000000,1.596288,0.000000,0.759861,0.000000
1,gpt4o_mini__semantic__clip_finetuned__reranker_1,gpt4o_mini,semantic,clip_finetuned,True,1293,0.921887,0.921887,0.000000,0.000000,...,0.825986,0.000000,0.888631,0.000000,0.904872,0.000000,1.596288,0.000000,0.759861,0.000000
2,gemini-2.5-flash__semantic__clip__reranker_1,gemini-2.5-flash,semantic,clip,True,1293,0.921114,0.921114,0.000000,0.000000,...,0.831400,0.000000,0.894045,0.000000,0.917247,0.000000,1.675174,0.000000,0.770946,0.000000
3,gemini-2.5-flash__semantic__clip_finetuned__re...,gemini-2.5-flash,semantic,clip_finetuned,True,1293,0.921114,0.921114,0.000000,0.000000,...,0.831400,0.000000,0.894045,0.000000,0.917247,0.000000,1.675174,0.000000,0.770946,0.000000
4,gemini-2.5-flash__fixed_size__clip__reranker_1,gemini-2.5-flash,fixed_size,clip,True,1293,0.895592,0.895592,0.000000,0.000000,...,0.821346,0.000000,0.889404,0.000000,0.908739,0.000000,1.685228,0.000000,0.760634,0.000000
5,gemini-2.5-flash__fixed_size__clip_finetuned__...,gemini-2.5-flash,fixed_size,clip_finetuned,True,1293,0.895592,0.895592,0.000000,0.000000,...,0.821346,0.000000,0.889404,0.000000,0.908739,0.000000,1.685228,0.000000,0.760634,0.000000
6,gpt4o_mini__fixed_size__clip__reranker_1,gpt4o_mini,fixed_size,clip,True,1293,0.890178,0.890178,0.000000,0.000000,...,0.815159,0.000000,0.880897,0.000000,0.901005,0.000000,1.638824,0.000000,0.755994,0.000000
7,gpt4o_mini__fixed_size__clip_finetuned__rerank...,gpt4o_mini,fixed_size,clip_finetuned,True,1293,0.890178,0.890178,0.000000,0.000000,...,0.815159,0.000000,0.880897,0.000000,0.901005,0.000000,1.638824,0.000000,0.755994,0.000000
8,gemini-2.5-flash__semantic__clip__reranker_0,gemini-2.5-flash,semantic,clip,False,1293,0.885538,0.885538,0.001547,0.001547,...,0.775715,0.002320,0.866976,0.020108,0.897138,0.030936,1.971384,0.282289,0.690255,0.000773
9,gemini-2.5-flash__semantic__clip_finetuned__re...,gemini-2.5-flash,semantic,clip_finetuned,False,1293,0.885538,0.885538,0.000773,0.000773,...,0.775715,0.000773,0.866203,0.006961,0.894818,0.010054,1.941222,0.096674,0.690255,0.000258


## Optional: Overwrite Original Summary Files

Run this cell only if you want the original filenames to include source metrics. The `_with_source_metrics.csv` files are already written above.

In [7]:
OVERWRITE_ORIGINALS = True

if OVERWRITE_ORIGINALS:
    updated_claim_metrics.to_csv(CLAIM_METRICS_CSV, index=False, encoding="utf-8-sig")
    updated_summary.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Updated original claim_metrics_long.csv and metrics_summary.csv")
else:
    print("Original files left unchanged")


Updated original claim_metrics_long.csv and metrics_summary.csv
